# Module 5 Guided Lab: Basic Data Transformation with Pandas

**BAN 6003: Data Management and Analytics Integration**

In this lab, we use the `nycflights13` dataset to practice the core transformation moves that analysts use all the time:

- filter rows
- select columns
- sort rows
- rename columns
- create derived columns
- export a transformed dataset

This module is intentionally focused. We are **not** doing aggregation, reshaping, joins, or heavy validation this week. Those topics come later.

## Module 5 Learning Goals

By the end of this guided lab, you should be able to:

1. Filter observations using `query()`.
2. Sort rows using `sort_values()`.
3. Select columns using `[[...]]`, `.loc[]`, and `.filter()`.
4. Rename variables using `.rename()`.
5. Create derived columns using `.assign()`.
6. Explain how a transformation changes the meaning or usability of a dataset.
7. Export a transformed dataset for later work.

The main idea is simple: **transformation changes raw columns into more useful analytical variables.**

## Part 0: Setup

We will use the Python version of the classic `nycflights13` dataset. It contains flight records from New York City airports in 2013.

If you are using **GitHub Codespaces**, this package should already be installed from `requirements.txt` when the Codespace is created.

If you are running the notebook **locally with Anaconda** and the import fails, install `nycflights13` directly into the Anaconda `base` environment by running the following command in a terminal:

```bash
conda run -n base python -m pip install nycflights13
```

You do **not** need to activate the `base` environment first. After the installation finishes, return to this notebook and rerun the import cell. If the import still fails, restart the notebook kernel and try again.


In [89]:
import nycflights13
import pandas as pd

print("Setup complete.")


Setup complete.


## Part 1: Load the Flights Dataset

The main table we will use is `flights`.

Each row represents one scheduled flight from a New York City airport in 2013. This row-level meaning matters. When we filter, sort, select, or create variables, we are still working at the **flight level**.

In [90]:
flights = nycflights13.flights.copy()

flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


In [91]:
flights.shape

(336776, 19)

In [92]:
flights.columns

Index(['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay',
       'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight',
       'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute',
       'time_hour'],
      dtype='str')

### Quick orientation

Before transforming anything, take a quick look at the variables. Some important columns for this lab are:

- `month`, `day`: date components
- `dep_time`, `arr_time`: actual departure and arrival times
- `dep_delay`, `arr_delay`: departure and arrival delays in minutes
- `carrier`: airline carrier code
- `origin`, `dest`: origin and destination airport codes
- `air_time`: flight time in minutes
- `distance`: distance in miles

We are not cleaning these variables this week. We are using them to practice transformation grammar.

In [93]:
flights[["month", "day", "dep_delay", "arr_delay", "carrier", "origin", "dest", "air_time", "distance"]].head()

,month,day,dep_delay,arr_delay,carrier,origin,dest,air_time,distance
0,1,1,2.0,11.0,UA,EWR,IAH,227.0,1400
1,1,1,4.0,20.0,UA,LGA,IAH,227.0,1416
2,1,1,2.0,33.0,AA,JFK,MIA,160.0,1089
3,1,1,-1.0,-18.0,B6,JFK,BQN,183.0,1576
4,1,1,-6.0,-25.0,DL,LGA,ATL,116.0,762


## Part 2: Filtering Rows with `query()`

Filtering means keeping only the rows that meet a condition.

In business analytics, filtering is how we narrow the data to a relevant scenario. For example:

- flights in January
- flights to Houston
- flights that arrived more than two hours late
- flights that left on time but arrived late

The `query()` method lets us write filtering conditions in a readable way.

In [94]:
# Flights in January
january_flights = flights.query("month == 1")

january_flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


In [95]:
# Flights on February 2
feb_2_flights = flights.query("month == 2 & day == 2")

feb_2_flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
112222,2013,2,2,3.0,2359,4.0,513.0,444,29.0,B6,739,N569JB,JFK,PSE,206.0,1617,23,59,2013-02-03T04:00:00Z
112223,2013,2,2,456.0,500,-4.0,640.0,648,-8.0,US,1117,N177US,EWR,CLT,84.0,529,5,0,2013-02-02T10:00:00Z
112224,2013,2,2,516.0,515,1.0,826.0,814,12.0,UA,785,N806UA,EWR,IAH,200.0,1400,5,15,2013-02-02T10:00:00Z
112225,2013,2,2,536.0,540,-4.0,838.0,850,-12.0,AA,1141,N638AA,JFK,MIA,159.0,1089,5,40,2013-02-02T10:00:00Z
112226,2013,2,2,543.0,530,13.0,835.0,829,6.0,UA,219,N490UA,LGA,IAH,212.0,1416,5,30,2013-02-02T10:00:00Z


In [96]:
# Flights on February 2 that departed late
feb_2_late_departures = flights.query("month == 2 & day == 2 & dep_delay > 0")

feb_2_late_departures.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
112222,2013,2,2,3.0,2359,4.0,513.0,444,29.0,B6,739,N569JB,JFK,PSE,206.0,1617,23,59,2013-02-03T04:00:00Z
112224,2013,2,2,516.0,515,1.0,826.0,814,12.0,UA,785,N806UA,EWR,IAH,200.0,1400,5,15,2013-02-02T10:00:00Z
112226,2013,2,2,543.0,530,13.0,835.0,829,6.0,UA,219,N490UA,LGA,IAH,212.0,1416,5,30,2013-02-02T10:00:00Z
112227,2013,2,2,549.0,540,9.0,1022.0,1017,5.0,B6,725,N558JB,JFK,BQN,196.0,1576,5,40,2013-02-02T10:00:00Z
112243,2013,2,2,601.0,600,1.0,930.0,927,3.0,UA,303,N532UA,JFK,SFO,371.0,2586,6,0,2013-02-02T11:00:00Z


### Common comparison operators in `query()`

| Operator | Meaning | Example |
|---|---|---|
| `==` | equals | `month == 1` |
| `!=` | not equal | `month != 12` |
| `>` | greater than | `arr_delay > 120` |
| `>=` | greater than or equal to | `arr_delay >= 120` |
| `<` | less than | `dep_delay < 0` |
| `<=` | less than or equal to | `dep_delay <= 0` |
| `in` | belongs to a list | `dest in ['IAH', 'HOU']` |

Use parentheses in your own thinking, even when Pandas does not require them, because filtering logic can get confusing quickly.

In [97]:
# Flights in November or December
holiday_season_flights = flights.query("month in [11, 12]")

holiday_season_flights.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
55893,2013,11,1,5.0,2359,6.0,352.0,345,7.0,B6,745,N568JB,JFK,PSE,205.0,1617,23,59,2013-11-02T03:00:00Z
55894,2013,11,1,35.0,2250,105.0,123.0,2356,87.0,B6,1816,N353JB,JFK,SYR,36.0,209,22,50,2013-11-02T02:00:00Z
55895,2013,11,1,455.0,500,-5.0,641.0,651,-10.0,US,1895,N192UW,EWR,CLT,88.0,529,5,0,2013-11-01T09:00:00Z
55896,2013,11,1,539.0,545,-6.0,856.0,827,29.0,UA,1714,N38727,LGA,IAH,229.0,1416,5,45,2013-11-01T09:00:00Z
55897,2013,11,1,542.0,545,-3.0,831.0,855,-24.0,AA,2243,N5CLAA,JFK,MIA,147.0,1089,5,45,2013-11-01T09:00:00Z


In [98]:
# Flights that arrived more than two hours late
very_late_arrivals = flights.query("arr_delay >= 120")

very_late_arrivals.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
119,2013,1,1,811.0,630,101.0,1047.0,830,137.0,MQ,4576,N531MQ,LGA,CLT,118.0,544,6,30,2013-01-01T11:00:00Z
151,2013,1,1,848.0,1835,853.0,1001.0,1950,851.0,MQ,3944,N942MQ,JFK,BWI,41.0,184,18,35,2013-01-01T23:00:00Z
218,2013,1,1,957.0,733,144.0,1056.0,853,123.0,UA,856,N534UA,EWR,BOS,37.0,200,7,33,2013-01-01T12:00:00Z
268,2013,1,1,1114.0,900,134.0,1447.0,1222,145.0,UA,1086,N76502,LGA,IAH,248.0,1416,9,0,2013-01-01T14:00:00Z
447,2013,1,1,1505.0,1310,115.0,1638.0,1431,127.0,EV,4497,N17984,EWR,RIC,63.0,277,13,10,2013-01-01T18:00:00Z


In [99]:
# Flights with missing tail numbers
missing_tailnum = flights.query("tailnum.isna()", engine="python")

missing_tailnum.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
1782,2013,1,2,NaN,1545,NaN,NaN,1910,NaN,AA,133,NaN,JFK,LAX,NaN,2475,15,45,2013-01-02T20:00:00Z
1784,2013,1,2,NaN,1601,NaN,NaN,1735,NaN,UA,623,NaN,EWR,ORD,NaN,719,16,1,2013-01-02T21:00:00Z
2697,2013,1,3,NaN,857,NaN,NaN,1209,NaN,UA,714,NaN,EWR,MIA,NaN,1085,8,57,2013-01-03T13:00:00Z
2698,2013,1,3,NaN,645,NaN,NaN,952,NaN,UA,719,NaN,EWR,DFW,NaN,1372,6,45,2013-01-03T11:00:00Z
3608,2013,1,4,NaN,845,NaN,NaN,1015,NaN,9E,3405,NaN,JFK,DCA,NaN,213,8,45,2013-01-04T13:00:00Z


### `query()` versus Boolean indexing

You may also see filtering written with Boolean indexing. This is equivalent to one of the examples above, but it is more verbose.

Both approaches are useful. In this course, `query()` is often easier to read for beginner-friendly filtering.

In [100]:
feb_2_late_departures_alt = flights[
    (flights["month"] == 2) &
    (flights["day"] == 2) &
    (flights["dep_delay"] > 0)
]

feb_2_late_departures_alt.head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
112222,2013,2,2,3.0,2359,4.0,513.0,444,29.0,B6,739,N569JB,JFK,PSE,206.0,1617,23,59,2013-02-03T04:00:00Z
112224,2013,2,2,516.0,515,1.0,826.0,814,12.0,UA,785,N806UA,EWR,IAH,200.0,1400,5,15,2013-02-02T10:00:00Z
112226,2013,2,2,543.0,530,13.0,835.0,829,6.0,UA,219,N490UA,LGA,IAH,212.0,1416,5,30,2013-02-02T10:00:00Z
112227,2013,2,2,549.0,540,9.0,1022.0,1017,5.0,B6,725,N558JB,JFK,BQN,196.0,1576,5,40,2013-02-02T10:00:00Z
112243,2013,2,2,601.0,600,1.0,930.0,927,3.0,UA,303,N532UA,JFK,SFO,371.0,2586,6,0,2013-02-02T11:00:00Z


### Your Turn 1: Filtering

Use `query()` to answer the following. For each task, show the number of matching rows using `.shape[0]`.

1. How many flights had an arrival delay of two or more hours?
2. How many flights flew to Houston, meaning destination is either `IAH` or `HOU`?
3. How many flights arrived more than two hours late but did not leave late?

In [101]:
# 1. Flights with arrival delay of two or more hours
# Your code here
flights[flights["arr_delay"] >= 120]


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
119,2013,1,1,811.0,630,101.0,1047.0,830,137.0,MQ,4576,N531MQ,LGA,CLT,118.0,544,6,30,2013-01-01T11:00:00Z
151,2013,1,1,848.0,1835,853.0,1001.0,1950,851.0,MQ,3944,N942MQ,JFK,BWI,41.0,184,18,35,2013-01-01T23:00:00Z
218,2013,1,1,957.0,733,144.0,1056.0,853,123.0,UA,856,N534UA,EWR,BOS,37.0,200,7,33,2013-01-01T12:00:00Z
268,2013,1,1,1114.0,900,134.0,1447.0,1222,145.0,UA,1086,N76502,LGA,IAH,248.0,1416,9,0,2013-01-01T14:00:00Z
447,2013,1,1,1505.0,1310,115.0,1638.0,1431,127.0,EV,4497,N17984,EWR,RIC,63.0,277,13,10,2013-01-01T18:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336579,2013,9,30,1823.0,1545,158.0,1934.0,1733,121.0,9E,3459,N916XJ,JFK,BNA,95.0,765,15,45,2013-09-30T19:00:00Z
336668,2013,9,30,1951.0,1649,182.0,2157.0,1903,174.0,EV,4294,N13988,EWR,SAV,95.0,708,16,49,2013-09-30T20:00:00Z
336724,2013,9,30,2053.0,1815,158.0,2310.0,2054,136.0,EV,5292,N600QX,EWR,ATL,91.0,746,18,15,2013-09-30T22:00:00Z
336757,2013,9,30,2159.0,1845,194.0,2344.0,2030,194.0,9E,3320,N906XJ,JFK,BUF,50.0,301,18,45,2013-09-30T22:00:00Z


In [102]:
# 2. Flights to Houston: IAH or HOU
# Your code here
flights[flights["dest"].isin(["IAH", "HOU"])]


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
32,2013,1,1,623.0,627,-4.0,933.0,932,1.0,UA,496,N459UA,LGA,IAH,229.0,1416,6,27,2013-01-01T11:00:00Z
81,2013,1,1,728.0,732,-4.0,1041.0,1038,3.0,UA,473,N488UA,LGA,IAH,238.0,1416,7,32,2013-01-01T12:00:00Z
89,2013,1,1,739.0,739,0.0,1104.0,1038,26.0,UA,1479,N37408,EWR,IAH,249.0,1400,7,39,2013-01-01T12:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336524,2013,9,30,1729.0,1720,9.0,2001.0,2010,-9.0,UA,652,N455UA,EWR,IAH,173.0,1400,17,20,2013-09-30T21:00:00Z
336527,2013,9,30,1735.0,1715,20.0,2010.0,2005,5.0,WN,2067,N296WN,EWR,HOU,188.0,1411,17,15,2013-09-30T21:00:00Z
336618,2013,9,30,1859.0,1859,0.0,2134.0,2159,-25.0,UA,1128,N14731,LGA,IAH,180.0,1416,18,59,2013-09-30T22:00:00Z
336694,2013,9,30,2015.0,2015,0.0,2244.0,2307,-23.0,UA,1545,N17730,EWR,IAH,174.0,1400,20,15,2013-10-01T00:00:00Z


In [103]:
# 3. Flights that arrived more than two hours late but did not leave late
# Your code here
flights[(flights["arr_delay"] >= 120) & (flights["dep_delay"] <= 0)]


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
22911,2013,1,27,1419.0,1420,-1.0,1754.0,1550,124.0,MQ,3728,N1EAMQ,EWR,ORD,135.0,719,14,20,2013-01-27T19:00:00Z
33011,2013,10,7,1350.0,1350,0.0,1736.0,1526,130.0,EV,5181,N611QX,LGA,MSN,117.0,812,13,50,2013-10-07T17:00:00Z
33019,2013,10,7,1357.0,1359,-2.0,1858.0,1654,124.0,AA,1151,N3CMAA,LGA,DFW,192.0,1389,13,59,2013-10-07T17:00:00Z
41075,2013,10,16,657.0,700,-3.0,1258.0,1056,122.0,B6,3,N703JB,JFK,SJU,225.0,1598,7,0,2013-10-16T11:00:00Z
55985,2013,11,1,658.0,700,-2.0,1329.0,1015,194.0,VX,399,N629VA,JFK,LAX,336.0,2475,7,0,2013-11-01T11:00:00Z
152766,2013,3,18,1844.0,1847,-3.0,39.0,2219,140.0,UA,389,N560UA,JFK,SFO,386.0,2586,18,47,2013-03-18T22:00:00Z
180893,2013,4,17,1635.0,1640,-5.0,2049.0,1845,124.0,MQ,4540,N721MQ,LGA,DTW,130.0,502,16,40,2013-04-17T20:00:00Z
181270,2013,4,18,558.0,600,-2.0,1149.0,850,179.0,AA,707,N3EXAA,LGA,DFW,234.0,1389,6,0,2013-04-18T10:00:00Z
181327,2013,4,18,655.0,700,-5.0,1213.0,950,143.0,AA,2083,N565AA,EWR,DFW,230.0,1372,7,0,2013-04-18T11:00:00Z
213693,2013,5,22,1827.0,1830,-3.0,2217.0,2010,127.0,MQ,4674,N518MQ,LGA,CLE,90.0,419,18,30,2013-05-22T22:00:00Z


## Part 3: Sorting Rows with `sort_values()`

Sorting does not remove rows. It changes the order so that important cases rise to the top.

This is useful when you want to inspect extremes, such as:

- the most delayed flights
- the earliest departures
- the longest flights
- the shortest flights

Sorting is a transformation because it changes how we review and interpret the data.

In [104]:
# Flights with the smallest departure delays first
flights.sort_values("dep_delay").head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
89673,2013,12,7,2040.0,2123,-43.0,40.0,2352,48.0,B6,97,N592JB,JFK,DEN,265.0,1626,21,23,2013-12-08T02:00:00Z
113633,2013,2,3,2022.0,2055,-33.0,2240.0,2338,-58.0,DL,1715,N612DL,LGA,MSY,162.0,1183,20,55,2013-02-04T01:00:00Z
64501,2013,11,10,1408.0,1440,-32.0,1549.0,1559,-10.0,EV,5713,N825AS,LGA,IAD,52.0,229,14,40,2013-11-10T19:00:00Z
9619,2013,1,11,1900.0,1930,-30.0,2233.0,2243,-10.0,DL,1435,N934DL,LGA,TPA,139.0,1010,19,30,2013-01-12T00:00:00Z
24915,2013,1,29,1703.0,1730,-27.0,1947.0,1957,-10.0,F9,837,N208FR,LGA,DEN,250.0,1620,17,30,2013-01-29T22:00:00Z


In [105]:
# Flights with the largest departure delays first
flights.sort_values("dep_delay", ascending=False).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
7072,2013,1,9,641.0,900,1301.0,1242.0,1530,1272.0,HA,51,N384HA,JFK,HNL,640.0,4983,9,0,2013-01-09T14:00:00Z
235778,2013,6,15,1432.0,1935,1137.0,1607.0,2120,1127.0,MQ,3535,N504MQ,JFK,CMH,74.0,483,19,35,2013-06-15T23:00:00Z
8239,2013,1,10,1121.0,1635,1126.0,1239.0,1810,1109.0,MQ,3695,N517MQ,EWR,ORD,111.0,719,16,35,2013-01-10T21:00:00Z
327043,2013,9,20,1139.0,1845,1014.0,1457.0,2210,1007.0,AA,177,N338AA,JFK,SFO,354.0,2586,18,45,2013-09-20T22:00:00Z
270376,2013,7,22,845.0,1600,1005.0,1044.0,1815,989.0,MQ,3075,N665MQ,JFK,CVG,96.0,589,16,0,2013-07-22T20:00:00Z


In [106]:
# Sort by multiple variables
# First by departure delay, then by arrival delay
flights.sort_values(["dep_delay", "arr_delay"]).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
89673,2013,12,7,2040.0,2123,-43.0,40.0,2352,48.0,B6,97,N592JB,JFK,DEN,265.0,1626,21,23,2013-12-08T02:00:00Z
113633,2013,2,3,2022.0,2055,-33.0,2240.0,2338,-58.0,DL,1715,N612DL,LGA,MSY,162.0,1183,20,55,2013-02-04T01:00:00Z
64501,2013,11,10,1408.0,1440,-32.0,1549.0,1559,-10.0,EV,5713,N825AS,LGA,IAD,52.0,229,14,40,2013-11-10T19:00:00Z
9619,2013,1,11,1900.0,1930,-30.0,2233.0,2243,-10.0,DL,1435,N934DL,LGA,TPA,139.0,1010,19,30,2013-01-12T00:00:00Z
24915,2013,1,29,1703.0,1730,-27.0,1947.0,1957,-10.0,F9,837,N208FR,LGA,DEN,250.0,1620,17,30,2013-01-29T22:00:00Z


In [107]:
# Mixed sort directions:
# departure delay descending, arrival delay ascending
flights.sort_values(["dep_delay", "arr_delay"], ascending=[False, True]).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
7072,2013,1,9,641.0,900,1301.0,1242.0,1530,1272.0,HA,51,N384HA,JFK,HNL,640.0,4983,9,0,2013-01-09T14:00:00Z
235778,2013,6,15,1432.0,1935,1137.0,1607.0,2120,1127.0,MQ,3535,N504MQ,JFK,CMH,74.0,483,19,35,2013-06-15T23:00:00Z
8239,2013,1,10,1121.0,1635,1126.0,1239.0,1810,1109.0,MQ,3695,N517MQ,EWR,ORD,111.0,719,16,35,2013-01-10T21:00:00Z
327043,2013,9,20,1139.0,1845,1014.0,1457.0,2210,1007.0,AA,177,N338AA,JFK,SFO,354.0,2586,18,45,2013-09-20T22:00:00Z
270376,2013,7,22,845.0,1600,1005.0,1044.0,1815,989.0,MQ,3075,N665MQ,JFK,CVG,96.0,589,16,0,2013-07-22T20:00:00Z


### Your Turn 2: Sorting

Use `sort_values()` to answer the following:

1. Which flights had the largest departure delays?
2. Which flights left earliest based on `dep_time`?
3. Which flights traveled the longest distance?
4. Which flights traveled the shortest distance?

Use `.head()` so the output is manageable.

In [108]:
# 1. Largest departure delays
# Your code here
flights.sort_values(["dep_delay"]).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
89673,2013,12,7,2040.0,2123,-43.0,40.0,2352,48.0,B6,97,N592JB,JFK,DEN,265.0,1626,21,23,2013-12-08T02:00:00Z
113633,2013,2,3,2022.0,2055,-33.0,2240.0,2338,-58.0,DL,1715,N612DL,LGA,MSY,162.0,1183,20,55,2013-02-04T01:00:00Z
64501,2013,11,10,1408.0,1440,-32.0,1549.0,1559,-10.0,EV,5713,N825AS,LGA,IAD,52.0,229,14,40,2013-11-10T19:00:00Z
9619,2013,1,11,1900.0,1930,-30.0,2233.0,2243,-10.0,DL,1435,N934DL,LGA,TPA,139.0,1010,19,30,2013-01-12T00:00:00Z
24915,2013,1,29,1703.0,1730,-27.0,1947.0,1957,-10.0,F9,837,N208FR,LGA,DEN,250.0,1620,17,30,2013-01-29T22:00:00Z


In [109]:
# 2. Earliest departure times
# Your code here
flights.sort_values(["dep_time"]).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
250450,2013,7,1,1.0,2029,212.0,236.0,2359,157.0,B6,915,N653JB,JFK,SFO,315.0,2586,20,29,2013-07-02T00:00:00Z
109552,2013,12,30,1.0,2359,2.0,441.0,437,4.0,B6,839,N508JB,JFK,BQN,198.0,1576,23,59,2013-12-31T04:00:00Z
240026,2013,6,20,1.0,2359,2.0,340.0,350,-10.0,B6,745,N517JB,JFK,PSE,196.0,1617,23,59,2013-06-21T03:00:00Z
212954,2013,5,22,1.0,1935,266.0,154.0,2140,254.0,EV,4361,N27200,EWR,TYS,94.0,631,19,35,2013-05-22T23:00:00Z
215892,2013,5,25,1.0,2359,2.0,336.0,341,-5.0,B6,727,N523JB,JFK,BQN,189.0,1576,23,59,2013-05-26T03:00:00Z


In [110]:
# 3. Longest distances
# Your code here
flights.sort_values(["distance"]).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
275945,2013,7,27,NaN,106,NaN,NaN,245,NaN,US,1632,NaN,EWR,LGA,NaN,17,1,6,2013-07-27T05:00:00Z
3083,2013,1,4,1240.0,1200,40.0,1333.0,1306,27.0,EV,4193,N14972,EWR,PHL,30.0,80,12,0,2013-01-04T17:00:00Z
16328,2013,1,19,1617.0,1617,0.0,1722.0,1722,0.0,EV,4616,N12540,EWR,PHL,34.0,80,16,17,2013-01-19T21:00:00Z
112178,2013,2,1,2128.0,2129,-1.0,2216.0,2224,-8.0,EV,4619,N13969,EWR,PHL,24.0,80,21,29,2013-02-02T02:00:00Z
19983,2013,1,23,2128.0,2129,-1.0,2221.0,2224,-3.0,EV,4619,N12135,EWR,PHL,23.0,80,21,29,2013-01-24T02:00:00Z


In [111]:
# 4. Shortest distances
# Your code here
flights.sort_values(["distance"], ascending=False).head()

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
50676,2013,10,26,1004.0,1000,4.0,1435.0,1450,-15.0,HA,51,N386HA,JFK,HNL,608.0,4983,10,0,2013-10-26T14:00:00Z
108078,2013,12,28,933.0,930,3.0,1520.0,1535,-15.0,HA,51,N384HA,JFK,HNL,633.0,4983,9,30,2013-12-28T14:00:00Z
100067,2013,12,19,924.0,930,-6.0,1450.0,1535,-45.0,HA,51,N386HA,JFK,HNL,609.0,4983,9,30,2013-12-19T14:00:00Z
179566,2013,4,16,953.0,1000,-7.0,1443.0,1510,-27.0,HA,51,N381HA,JFK,HNL,631.0,4983,10,0,2013-04-16T14:00:00Z
30229,2013,10,4,954.0,1000,-6.0,1438.0,1450,-12.0,HA,51,N380HA,JFK,HNL,618.0,4983,10,0,2013-10-04T14:00:00Z


## Part 4: Selecting Columns

Selecting columns means keeping only the variables you need.

This is an important transformation habit because real datasets often have many columns. A smaller working DataFrame is easier to read, easier to debug, and easier to explain.

### Single brackets versus double brackets

A very common beginner issue is the difference between selecting one column as a Series and selecting one column as a DataFrame.

In [112]:
one_column_series = flights["year"]

type(one_column_series)

pandas.Series

In [113]:
one_column_dataframe = flights[["year"]]

type(one_column_dataframe)

pandas.DataFrame

### Selecting multiple columns

To select multiple columns, use a list of column names inside double brackets.

In [114]:
date_delay_columns = flights[["year", "month", "day", "dep_delay", "arr_delay"]]

date_delay_columns.head()

,year,month,day,dep_delay,arr_delay
0,2013,1,1,2.0,11.0
1,2013,1,1,4.0,20.0
2,2013,1,1,2.0,33.0
3,2013,1,1,-1.0,-18.0
4,2013,1,1,-6.0,-25.0


In [115]:
# Equivalent selection using .loc[row_selection, column_selection]
date_delay_columns_loc = flights.loc[:, ["year", "month", "day", "dep_delay", "arr_delay"]]

date_delay_columns_loc.head()

,year,month,day,dep_delay,arr_delay
0,2013,1,1,2.0,11.0
1,2013,1,1,4.0,20.0
2,2013,1,1,2.0,33.0
3,2013,1,1,-1.0,-18.0
4,2013,1,1,-6.0,-25.0


### Finding columns by keyword

The `.filter(like=...)` method is useful when column names share a pattern.

For example, many variables related to departure include `"dep"` in the name.

In [116]:
flights.filter(like="dep").head()

,dep_time,sched_dep_time,dep_delay
0,517.0,515,2.0
1,533.0,529,4.0
2,542.0,540,2.0
3,544.0,545,-1.0
4,554.0,600,-6.0


### Selecting from a requested list safely

Sometimes you have a list of desired columns, but not every name exists in the DataFrame. The `.columns.intersection()` approach keeps only names that actually exist.

This is useful when you are working with changing datasets or user-provided column lists.

In [117]:
requested_columns = ["MONTH", "month", "day", "dep_delay", "arr_delay"]

available_columns = flights.columns.intersection(requested_columns)

available_columns

Index(['month', 'day', 'dep_delay', 'arr_delay'], dtype='str')

In [118]:
flights[available_columns].head()

,month,day,dep_delay,arr_delay
0,1,1,2.0,11.0
1,1,1,4.0,20.0
2,1,1,2.0,33.0
3,1,1,-1.0,-18.0
4,1,1,-6.0,-25.0


### Your Turn 3: Selecting Columns

1. What happens if you include the same variable more than once inside double brackets?
2. Select the columns included in this list, but only if they exist in the dataset:

```python
var_list = ["MONTH", "month", "day", "dep_delay", "arr_delay"]
```

3. Select all columns that contain `"time"` in the name.

In [119]:
# 1. Include a variable multiple times
# Your code here
flights[["arr_delay", "dep_delay", "distance"]]

,arr_delay,dep_delay,distance
0,11.0,2.0,1400
1,20.0,4.0,1416
2,33.0,2.0,1089
3,-18.0,-1.0,1576
4,-25.0,-6.0,762
...,...,...,...
336771,NaN,NaN,213
336772,NaN,NaN,198
336773,NaN,NaN,764
336774,NaN,NaN,419


In [120]:
# 2. Select only available variables from the list
var_list = ["MONTH", "month", "day", "dep_delay", "arr_delay"]

# Your code here
flights[[var for var in var_list if var in flights.columns]]

,month,day,dep_delay,arr_delay
0,1,1,2.0,11.0
1,1,1,4.0,20.0
2,1,1,2.0,33.0
3,1,1,-1.0,-18.0
4,1,1,-6.0,-25.0
...,...,...,...,...
336771,9,30,NaN,NaN
336772,9,30,NaN,NaN
336773,9,30,NaN,NaN
336774,9,30,NaN,NaN


In [121]:
# 3. Select variables that contain "time" in the name
# Your code here
flights[[var for var in var_list if "time" in var]]

""
0
1
2
3
4
...
336771
336772
336773
336774


## Part 5: Renaming Columns

Renaming is a small change that can make a dataset much easier to understand.

In practice, raw variable names are often abbreviated or inconsistent. We do not always rename every variable, but we often rename key variables to make later notebooks and reports clearer.

In [122]:
flight_sml = flights[["year", "month", "day", "arr_delay", "dep_delay", "air_time", "distance"]].copy()

flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance
0,2013,1,1,11.0,2.0,227.0,1400
1,2013,1,1,20.0,4.0,227.0,1416
2,2013,1,1,33.0,2.0,160.0,1089
3,2013,1,1,-18.0,-1.0,183.0,1576
4,2013,1,1,-25.0,-6.0,116.0,762


In [123]:
flight_sml = flight_sml.rename(columns={
    "arr_delay": "arrival_delay",
    "dep_delay": "departure_delay"
})

flight_sml.head()

,year,month,day,arrival_delay,departure_delay,air_time,distance
0,2013,1,1,11.0,2.0,227.0,1400
1,2013,1,1,20.0,4.0,227.0,1416
2,2013,1,1,33.0,2.0,160.0,1089
3,2013,1,1,-18.0,-1.0,183.0,1576
4,2013,1,1,-25.0,-6.0,116.0,762


### Naming tip

Use clear `snake_case` names for new columns:

- good: `arrival_delay`, `departure_delay`, `speed_mph`
- harder to read: `arrDelay`, `x1`, `newvar`, `delay2`

Good names help future you, your teammates, and your instructor understand your work.

## Part 6: Creating Derived Columns with `assign()`

Derived columns are new variables created from existing variables.

This is one of the most important transformation skills in analytics. Derived variables often become features for modeling, KPIs for reporting, or business indicators for decision-making.

For this lab, we will create:

- `delay_change_minutes`: arrival delay minus departure delay; positive values mean the flight became more delayed, while negative values mean it recovered time
- `air_time_hours`: air time converted from minutes to hours
- `speed_mph`: distance divided by air time in hours

The lecture used the shorter label `delay_gain` for the same arrival-minus-departure calculation. Here we use `delay_change_minutes` so the direction and unit are explicit.

In [124]:
flight_sml = flights[["year", "month", "day", "arr_delay", "dep_delay", "air_time", "distance"]].copy()

flight_sml = flight_sml.assign(
    delay_change_minutes = flight_sml["arr_delay"] - flight_sml["dep_delay"],
    air_time_hours = flight_sml["air_time"] / 60
)

flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance,delay_change_minutes,air_time_hours
0,2013,1,1,11.0,2.0,227.0,1400,9.0,3.783333
1,2013,1,1,20.0,4.0,227.0,1416,16.0,3.783333
2,2013,1,1,33.0,2.0,160.0,1089,31.0,2.666667
3,2013,1,1,-18.0,-1.0,183.0,1576,-17.0,3.050000
4,2013,1,1,-25.0,-6.0,116.0,762,-19.0,1.933333


In [125]:
flight_sml = flight_sml.assign(
    speed_mph = flight_sml["distance"] / flight_sml["air_time_hours"]
)

flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance,delay_change_minutes,air_time_hours,speed_mph
0,2013,1,1,11.0,2.0,227.0,1400,9.0,3.783333,370.044053
1,2013,1,1,20.0,4.0,227.0,1416,16.0,3.783333,374.273128
2,2013,1,1,33.0,2.0,160.0,1089,31.0,2.666667,408.375000
3,2013,1,1,-18.0,-1.0,183.0,1576,-17.0,3.050000,516.721311
4,2013,1,1,-25.0,-6.0,116.0,762,-19.0,1.933333,394.137931


### Your Turn 4: Creating Derived Columns

Create a smaller DataFrame called `flight_efficiency` that contains:

- `air_time`
- `distance`

Then create:

1. `distance_km`, converting distance from miles to kilometers using `1 mile = 1.61 kilometers`.
2. `minutes_per_km`, using `air_time / distance_km`.

Show the first few rows.

In [126]:
# Create flight_efficiency with air_time and distance
# Your code here
flight_sml = flight_sml.assign(
    flight_efficiency = flight_sml["air_time"] / flight_sml["distance"]
)
flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance,delay_change_minutes,air_time_hours,speed_mph,flight_efficiency
0,2013,1,1,11.0,2.0,227.0,1400,9.0,3.783333,370.044053,0.162143
1,2013,1,1,20.0,4.0,227.0,1416,16.0,3.783333,374.273128,0.160311
2,2013,1,1,33.0,2.0,160.0,1089,31.0,2.666667,408.375000,0.146924
3,2013,1,1,-18.0,-1.0,183.0,1576,-17.0,3.050000,516.721311,0.116117
4,2013,1,1,-25.0,-6.0,116.0,762,-19.0,1.933333,394.137931,0.152231


In [127]:
# Add distance_km
# Your code here
flight_sml = flight_sml.assign(
    distance_km = flight_sml["distance"] * 1.60934
)
flight_sml.head()



,year,month,day,arr_delay,dep_delay,air_time,distance,delay_change_minutes,air_time_hours,speed_mph,flight_efficiency,distance_km
0,2013,1,1,11.0,2.0,227.0,1400,9.0,3.783333,370.044053,0.162143,2253.07600
1,2013,1,1,20.0,4.0,227.0,1416,16.0,3.783333,374.273128,0.160311,2278.82544
2,2013,1,1,33.0,2.0,160.0,1089,31.0,2.666667,408.375000,0.146924,1752.57126
3,2013,1,1,-18.0,-1.0,183.0,1576,-17.0,3.050000,516.721311,0.116117,2536.31984
4,2013,1,1,-25.0,-6.0,116.0,762,-19.0,1.933333,394.137931,0.152231,1226.31708


In [128]:
# Add minutes_per_km
# Your code here
flight_sml = flight_sml.assign(
    minutes_per_km = flight_sml["air_time"] / flight_sml["distance_km"]
)
flight_sml.head()

,year,month,day,arr_delay,dep_delay,air_time,distance,delay_change_minutes,air_time_hours,speed_mph,flight_efficiency,distance_km,minutes_per_km
0,2013,1,1,11.0,2.0,227.0,1400,9.0,3.783333,370.044053,0.162143,2253.07600,0.100751
1,2013,1,1,20.0,4.0,227.0,1416,16.0,3.783333,374.273128,0.160311,2278.82544,0.099613
2,2013,1,1,33.0,2.0,160.0,1089,31.0,2.666667,408.375000,0.146924,1752.57126,0.091294
3,2013,1,1,-18.0,-1.0,183.0,1576,-17.0,3.050000,516.721311,0.116117,2536.31984,0.072152
4,2013,1,1,-25.0,-6.0,116.0,762,-19.0,1.933333,394.137931,0.152231,1226.31708,0.094592


## Part 7: Put the Transformation Steps Together

Scenario: an operations analyst needs a reusable row-level file for flights departing JFK during June, July, and August. The file should support later delay review without introducing aggregation yet.

We will:

1. filter to the requested flights and require usable air time and distance;
2. select and rename relevant columns;
3. create clearly defined derived columns;
4. sort the most delayed arrivals first; and
5. export the transformed dataset.

Each row will still represent one flight.

In [129]:
# Step 1: filter to the business scope and usable flight records
transformed_flights = flights.query(
    "origin == 'JFK' & month in [6, 7, 8] & air_time.notna() & distance.notna()",
    engine="python",
)

# Step 2: select useful columns
transformed_flights = transformed_flights[[
    "year", "month", "day", "carrier", "origin", "dest",
    "dep_delay", "arr_delay", "air_time", "distance"
]]

# Step 3: rename columns for clarity
transformed_flights = transformed_flights.rename(columns={
    "dep_delay": "departure_delay",
    "arr_delay": "arrival_delay"
})

# Step 4: create derived columns with explicit units and direction
transformed_flights = transformed_flights.assign(
    delay_change_minutes=(
        transformed_flights["arrival_delay"] - transformed_flights["departure_delay"]
    ),
    air_time_hours=transformed_flights["air_time"] / 60,
    speed_mph=(
        transformed_flights["distance"] / (transformed_flights["air_time"] / 60)
    ),
)

# Step 5: put the most delayed arrivals first
transformed_flights = transformed_flights.sort_values(
    "arrival_delay", ascending=False
).reset_index(drop=True)

transformed_flights.head()

,year,month,day,carrier,origin,dest,departure_delay,arrival_delay,air_time,distance,delay_change_minutes,air_time_hours,speed_mph
0,2013,6,15,MQ,JFK,CMH,1137.0,1127.0,74.0,483,-10.0,1.233333,391.621622
1,2013,7,22,MQ,JFK,CVG,1005.0,989.0,96.0,589,-16.0,1.600000,368.125000
2,2013,6,27,DL,JFK,PDX,899.0,850.0,313.0,2454,-49.0,5.216667,470.415335
3,2013,6,27,DL,JFK,SAN,790.0,769.0,312.0,2446,-21.0,5.200000,470.384615
4,2013,7,7,VX,JFK,SFO,629.0,676.0,334.0,2586,47.0,5.566667,464.550898


### Short Reflection

Write 3-4 sentences below.

1. What does one row in `transformed_flights` represent, and which flights are excluded by the business filter?
2. What does a positive versus negative `delay_change_minutes` value mean?
3. Why is `speed_mph` only an approximate airborne speed?

This is not a full transformation summary yet. For now, practice explaining what the transformed file can and cannot support.

**Your response here:**

- It represents the flights with "origin == 'JFK' during the months [6, 7, 8] june, july and august. So the flights withouth this characteristics wold be excluded.
- positive delay change minutes: that they didn't depart on time and negative that they did depart on time  
- speed_mph = distance / air_time_hours it's just an average of the plane speed, because we don't have the precise data since it changes 

## Part 8: Export the Transformed Dataset

Exporting gives us a saved output that can be reused later.

For now, we will export a CSV file. Later in the course, we will discuss other formats and database storage.

In [130]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "flights_transformed_module5.csv"
transformed_flights.to_csv(output_path, index=False)

print(f"Saved {len(transformed_flights):,} rows to: {output_path}")

Saved 28,809 rows to: ../data/processed/flights_transformed_module5.csv


## Part 9: Offline Assignment - Operational Delay Review

Complete this task independently using `transformed_flights`.

An operations manager wants a compact review file containing the 50 JFK summer flights with the largest arrival delays among flights delayed by at least 60 minutes.

1. Filter `transformed_flights` to `arrival_delay >= 60`.
2. Select `year`, `month`, `day`, `carrier`, `dest`, `departure_delay`, `arrival_delay`, and `delay_change_minutes`.
3. Sort by `arrival_delay` from largest to smallest and keep the first 50 rows.
4. Save the result as `data/processed/jfk_summer_delay_review_top50.csv`.
5. Display the first five rows and the final shape.

In [131]:
# Offline Assignment - Your code here
# Suggested sequence:
# 1. Use .query() for the delay condition.

flight_sml = flight_sml.query("delay > 60")

flight_sml.head()

# 2. Select the eight requested columns with [[...]].
flight_sml = flight_sml[[ "year", "month", "day", "carrier","dest", "departure_delay","arrival_delay","delay_change_minutes"]]
# 3. Use .sort_values(..., ascending=False).head(50).
flight_sml = flight_sml.sort_values(by="delay_change_minutes", ascending=False).head(50)
# 4. Save with output_dir / "jfk_summer_delay_review_top50.csv".
flight_sml.to_csv(output_dir / "jfk_summer_delay_review_top50.csv", index=False)
# Build a DataFrame named jfk_summer_delay_review.
jfk_summer_delay_review = flight_sml.copy()


UndefinedVariableError: name 'delay' is not defined

### Offline Assignment Reflection

**Your response here:** Write 4-5 sentences explaining what one row represents, why this file is not representative of all flights, what `delay_change_minutes` adds to the review, and one question the data cannot answer by itself.

 onw row represents one flight, this file is focused on nyc flights from 2013, and whenever you filter through `delay_change_minutes` is a way to see which and how many flights departed late from the airport. A more specific question in the file that data cannot answer would be how many minutes each flight was delayed or also, what was the exact speed the flights had during the whole flights.

## Lab Wrap-Up

In this lab, you practiced the core Pandas transformation moves:

- `query()` for filtering rows
- `sort_values()` for ordering rows
- `[[...]]`, `.loc[]`, and `.filter()` for selecting columns
- `rename()` for clearer variable names
- `assign()` for derived columns
- `to_csv()` for exporting a transformed dataset

Before submitting, complete the offline assignment, rerun the notebook from the top, and confirm that both requested outputs are in `data/processed`.

Next module, we will focus on **string and categorical data transformation**.